# PRO-cap Atlas BPNet locus viewer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kundajelab/procap-atlas/blob/main/notebooks/procap_atlas_bpnet_locus_viewer.ipynb)

This lightweight notebook visualizes one locus with fold-averaged BPNet predictions and DeepLIFT/SHAP logos. It is set up to run in Google Colab by default. Sherlock/Open OnDemand users can skip the Colab bootstrap and optionally run the environment check below.


## Colab setup

Run this cell first in Colab. It clones the repository into `/content/procap-atlas`, installs the notebook runtime dependencies, and changes the working directory to the checkout. Outside Colab, it leaves the current checkout alone.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = Path("/content/procap-atlas") if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/kundajelab/procap-atlas.git",
                str(REPO_DIR),
            ],
            check=True,
        )
    os.chdir(REPO_DIR)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "bpnet-lite",
            "huggingface-hub",
            "matplotlib",
            "numpy",
            "pandas",
            "pybigtools==0.2.5",
            "pyfaidx",
            "pyyaml",
            "seaborn",
            "tangermeme",
        ],
        check=True,
    )

print(f"Running in Colab: {IN_COLAB}")
print(f"Working directory: {Path.cwd()}")


## Optional Open OnDemand check

Sherlock Open OnDemand users can set `RUN_ONDEMAND_ENV_CHECK = True` and run this cell after selecting the `PRO-cap Atlas (uv)` kernel. Colab users should leave it disabled.


In [ ]:
RUN_ONDEMAND_ENV_CHECK = False

if RUN_ONDEMAND_ENV_CHECK:
    from pathlib import Path as _Path
    import importlib.util as _importlib_util
    import os as _os
    import sys as _sys

    _numpy_spec = _importlib_util.find_spec("numpy")
    _numpy_origin = _Path(_numpy_spec.origin).resolve() if _numpy_spec else None
    _environment_root = _Path(_sys.prefix).resolve()
    print(f"Python: {_sys.executable}")
    print(f"Environment: {_environment_root}")
    print(f"NumPy candidate: {_numpy_origin}")
    print(f"PYTHONPATH: {_os.environ.get('PYTHONPATH')!r}")
    if _os.environ.get("PYTHONPATH"):
        raise RuntimeError(
            "The PRO-cap Atlas kernel did not remove the Open OnDemand PYTHONPATH. "
            "Reinstall it with notebooks/install_uv_kernel.py and restart JupyterLab."
        )
    _ondemand_paths = [
        path
        for path in _sys.path
        if path.startswith("/share/software/user/open/py-jupyterlab/")
    ]
    if _ondemand_paths:
        raise RuntimeError(
            f"Open OnDemand PYTHONPATH entries are affecting imports: {_ondemand_paths}. "
            "Reinstall the kernel with notebooks/install_uv_kernel.py."
        )
    if _numpy_origin is None or not _numpy_origin.is_relative_to(_environment_root):
        raise RuntimeError(
            "NumPy is not resolving from the selected uv environment. Reinstall the "
            "PRO-cap Atlas kernel with notebooks/install_uv_kernel.py."
        )
else:
    print("Skipping Open OnDemand environment check.")


## Import dependencies

In [ ]:
from pathlib import Path
import importlib
import os
import sys

import matplotlib.pyplot as plt
import seaborn as sns
import torch

for _parent in [Path.cwd(), *Path.cwd().parents]:
    if (_parent / "src").is_dir() and str(_parent) not in sys.path:
        sys.path.insert(0, str(_parent))
        break

import notebooks.locus_viewer as locus_viewer

locus_viewer = importlib.reload(locus_viewer)

from notebooks.locus_viewer import (
    deeplift_attributions,
    ensemble_prediction,
    logo_offsets_for_region,
    plot_locus_summary,
    region_input,
    save_locus_viewer_outputs,
    setup_experiment,
)

sns.set_style("whitegrid")
_ipython = globals().get("get_ipython", lambda: None)()
if _ipython is not None:
    try:
        _ipython.run_line_magic(
            "config", "InlineBackend.print_figure_kwargs = {'bbox_inches': None}"
        )
    except Exception:
        pass


## Configuration

Choose the experiment and one 1-based region of interest. The model input is centered on this region, and the same region is used for the observed/predicted tracks and DeepLIFT logos. Set `TRACK_VALUE_TRANSFORM` to `None`, `"sqrt"`, or `"log1p"` to control display-only squashing of observed and predicted signal values.


In [ ]:
EXP_ID = "ENCSR342WAR"
REGION = "chr2:181680467-181681166"
REVERSE_COMPLEMENT = False
TRACK_VALUE_TRANSFORM = None
N_FOLDS = 7
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_DIR = Path(os.environ.get("SCRATCH", ".cache")) / "procap_atlas_locus_viewer"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Work directory: {WORK_DIR}")


The following helper functions will download and set up the data and model files.

In [ ]:
resources = setup_experiment(EXP_ID, WORK_DIR, N_FOLDS)
chrom, region_start, region_end, center, X = region_input(resources, REGION)
logo_chrom, logo_start, logo_end, logo_offsets = logo_offsets_for_region(REGION)
print(resources["config"].get("biosample", EXP_ID), X.shape, logo_offsets)


## Predictions

Run each fold locally and average the count-scaled strand profiles. The observed and predicted tracks are plotted separately in the combined summary figure below so each can use its own y scale.


In [ ]:
prediction = ensemble_prediction(resources, X, N_FOLDS, DEVICE)


## DeepLIFT

Compute profile-head and count-head DeepLIFT/SHAP logos using one soft reference whose A/C/G/T probabilities are the observed input-wide nucleotide frequencies.

In [ ]:
attributions = deeplift_attributions(
    resources, X, logo_offsets, N_FOLDS, BATCH_SIZE, DEVICE
)


## Summary figure

The observed track, predicted track, and logos are stacked in one figure to make the panels easier to compare.

In [ ]:
plot_locus_summary(
    prediction,
    attributions,
    resources,
    EXP_ID,
    REGION,
    REGION,
    REGION,
    logo_start,
    logo_end,
    REVERSE_COMPLEMENT,
    TRACK_VALUE_TRANSFORM,
)


## Optional save

Save the current figures and arrays for later inspection.


In [ ]:
OUTPUT_DIR = (
    Path("plots/bpnet/locus_viewer")
    / EXP_ID
    / REGION.replace(":", "_").replace(",", "")
)
save_locus_viewer_outputs(
    OUTPUT_DIR,
    prediction,
    attributions,
    resources,
    EXP_ID,
    REGION,
    REGION,
    REGION,
    logo_start,
    logo_end,
    REVERSE_COMPLEMENT,
    TRACK_VALUE_TRANSFORM,
)
print(f"Saved {OUTPUT_DIR}")
